# LISA MBHB detection CNN (Houba et al. 2024, detection stage only)

Recreates **Section III** of the paper: a CNN that looks at a *time-evolving TDI spectrogram* (10 days of data) and outputs a single number. **Positive = signal detected, negative = no signal.** The reinforcement-learning / actor-critic merger-time predictor (Sections IV-VI) is intentionally left out.

**Pipeline**
1. Simulate training data (inspiral chirp + LISA-like colored noise) -> whitened log spectrograms (128x128).
2. Train the paper's CNN (2xConv32, 2xConv64, 2xConv64, 3 dense layers, 1 linear output).
3. Evaluate detection rate vs. time-to-merger.
4. Run on LISA Spritz (sliding window).

Set `CHANNELS = ["A"]` for the paper's setup or `["A","E"]` for your A+E extension. Everything else adapts automatically.

**Honest caveats:** the paper uses `lisagwresponse` + `pyTDI` with IMRPhenomD to build TDI data. Here a lightweight Newtonian-chirp surrogate and an analytic noise curve stand in so it runs fast in Colab. The CNN, input format, labels and detection rule follow the paper.

In [ ]:
!pip -q install scikit-image
import numpy as np, matplotlib.pyplot as plt, time
from scipy import signal
import tensorflow as tf
from tensorflow.keras import layers, models
print(tf.__version__, "GPU:", tf.config.list_physical_devices("GPU"))

## 1. Configuration

In [ ]:
CHANNELS   = ["A"]          # ["A"] = paper;  ["A","E"] = your extension
FS         = 0.2            # Hz (5 s cadence, same as LISA Spritz)
WINDOW_D   = 10             # days per spectrogram (paper: 10 days)
N          = int(WINDOW_D*86400*FS)   # samples per window
NPERSEG, NOVERLAP = 2048, 1792
IMG        = 128            # image is IMG x IMG (paper uses reduced resolution too)
FMIN, FMAX = 1e-4, 1e-1
N_TRAIN, N_TEST = 6000, 1500   # half signal, half noise-only (paper: 5M train; raise if you have time)
SEED = 0
rng = np.random.default_rng(SEED)
dt = 1/FS

## 2. Noise model and signal surrogate

In [ ]:
L, fstar = 2.5e9, 19.09e-3
def Sn(f):
    """Approximate single-detector LISA strain noise PSD (1/Hz). Analytic stand-in for the science-requirement noise."""
    f = np.maximum(f, 1e-5)
    Poms = (1.5e-11)**2*(1+(2e-3/f)**4)
    Pacc = (3e-15)**2*(1+(0.4e-3/f)**2)*(1+(f/8e-3)**4)
    return (1/L**2)*(Poms + 2*(1+np.cos(f/fstar)**2)*Pacc/(2*np.pi*f)**4)*(1+0.6*(f/fstar)**2)/0.75

freqs_rfft = np.fft.rfftfreq(N, dt)
SN_RFFT = Sn(freqs_rfft)

def colored_noise():
    k = len(freqs_rfft)
    X = (rng.standard_normal(k)+1j*rng.standard_normal(k))/np.sqrt(2)*np.sqrt(N*SN_RFFT/(2*dt))
    X[0] = 0; X[-1] = X[-1].real
    return np.fft.irfft(X, n=N)

# --- chirp surrogate (leading-order inspiral, truncated at ISCO) ---
MSUN_S = 4.925491e-6        # G Msun / c^3 [s]
GPC_S  = 3.0857e25/2.998e8  # 1 Gpc in light-seconds
def chirp(m1, q, D_gpc, tau_end, incl, F):
    """Returns list of per-channel strain series for a window ending tau_end seconds before merger."""
    m2 = q*m1; M = (m1+m2)*MSUN_S
    Mc = ((m1*m2)**0.6/(m1+m2)**0.2)*MSUN_S
    f_isco = 1/(6**1.5*np.pi*M)
    tau_isco = 5/256*(np.pi*f_isco)**(-8/3)*Mc**(-5/3)
    tm = tau_end + (N-1-np.arange(N))*dt          # time to "merger" (ISCO) for each sample
    tauN = tm + tau_isco
    f = (1/np.pi)*(5/(256*tauN))**(3/8)*Mc**(-5/8)
    amp = 4*Mc**(5/3)*(np.pi*f)**(2/3)/(D_gpc*GPC_S)
    # taper the first 2 hours so the window start doesn't ring
    ramp = np.clip(np.arange(N)*dt/7200, 0, 1)
    phase = 2*np.pi*np.cumsum(f)*dt
    hp = amp*(1+np.cos(incl)**2)/2*np.cos(phase)*ramp
    hx = amp*np.cos(incl)*np.sin(phase)*ramp
    return [Fp*hp+Fx*hx for (Fp, Fx) in F]

def antenna(phi, u, psi):
    Fp = 0.5*(1+u**2)*np.cos(2*phi)*np.cos(2*psi) - u*np.sin(2*phi)*np.sin(2*psi)
    Fx = 0.5*(1+u**2)*np.cos(2*phi)*np.sin(2*psi) + u*np.sin(2*phi)*np.cos(2*psi)
    return Fp, Fx

def optimal_snr(h):
    H = dt*np.fft.rfft(h); df = 1/(N*dt)
    return np.sqrt(4*np.sum(np.abs(H[1:])**2/SN_RFFT[1:])*df)

# quick sanity check of the surrogate
h = chirp(3e5, 1/3, 15, 86400, 0.8, [antenna(0.3, 0.4, 0.5)])[0]
print("example SNR in a window ending 1 day before merger (3e5 Msun, 15 Gpc):", round(optimal_snr(h),1))

## 3. Spectrograms
Whitening divides by the noise PSD so pure noise has power ~1 in every pixel. Frequency axis is log-spaced (1e-4 to 1e-1 Hz) like the paper's plots.

In [ ]:
_f = None
def spec_whitened(x, ref_psd=None):
    global _f
    f, t, P = signal.spectrogram(x, FS, window="hann", nperseg=NPERSEG, noverlap=NOVERLAP, scaling="density", detrend=False)
    _f = f
    w = Sn(f) if ref_psd is None else ref_psd
    return f, P/w[:, None]

LOGF = np.logspace(np.log10(FMIN), np.log10(FMAX), IMG)
def to_image(f, Pw, log=True):
    from scipy.interpolate import interp1d
    Pw = interp1d(f[1:], Pw[1:], axis=0, kind="linear", bounds_error=False, fill_value="extrapolate")(LOGF)
    Pw = Pw[:, -IMG*5:]                                   # keep most recent 640 columns
    Pw = Pw.reshape(IMG, IMG, 5).mean(-1)                 # 640 -> 128 columns
    Pw = np.flipud(Pw)                                    # low frequency at the bottom
    if log:
        Pw = np.log10(np.maximum(Pw, 1e-6))
        Pw = (Pw-Pw.mean())/(Pw.std()+1e-9)
    return Pw.astype(np.float32)

def trace_label(img_clean):
    """Mean vertical pixel position (scaled to (0,1]) of the signal trace in the noise-free spectrogram.
    Simple threshold stand-in for the paper's Canny edge detection."""
    m = img_clean > 0.05*img_clean.max()
    rows = np.where(m.any(1))[0]
    # row 0 = top of image; paper's 'vertical pixel position' is just a positive number
    return float((np.mean(np.where(m)[0])+1)/IMG)

def make_sample(has_signal):
    chans = [colored_noise() for _ in CHANNELS]
    meta = dict(tau=np.nan, snr=0.0)
    label = -1.0
    if has_signal:
        m1  = 10**rng.uniform(5, 7); q = rng.uniform(0.1, 1.0)
        D   = rng.uniform(1, 100)
        tau = 10**rng.uniform(np.log10(300), np.log10(30*86400))  # 5 min .. 30 days before merger
        incl = np.arccos(rng.uniform(-1, 1))
        u, phi, psi = rng.uniform(-1, 1), rng.uniform(0, 2*np.pi), rng.uniform(0, np.pi)
        F = [antenna(phi, u, psi), antenna(phi-np.pi/4, u, psi)][:len(CHANNELS)]   # E = A rotated by 45 deg
        h = chirp(m1, q, D, tau, incl, F)
        clean = []
        for i in range(len(CHANNELS)):
            chans[i] = chans[i] + h[i]
            f, Pc = spec_whitened(h[i]); clean.append(to_image(f, Pc, log=False))
        label = trace_label(np.mean(clean, 0))   # same label for all channels
        meta = dict(tau=tau, snr=float(np.sqrt(sum(optimal_snr(x)**2 for x in h))))
    imgs = []
    for x in chans:
        f, Pw = spec_whitened(x); imgs.append(to_image(f, Pw))
    return np.stack(imgs, -1), label, meta

def make_dataset(n):
    X = np.zeros((n, IMG, IMG, len(CHANNELS)), np.float32); y = np.zeros(n, np.float32); metas = []
    flags = np.arange(n) % 2 == 0
    t0 = time.time()
    for i in range(n):
        X[i], y[i], m = make_sample(flags[i]); metas.append(m)
        if (i+1) % 500 == 0: print(f"  {i+1}/{n}  ({time.time()-t0:.0f}s)")
    p = rng.permutation(n)
    return X[p], y[p], [metas[j] for j in p]

# look at a few examples first
fig, ax = plt.subplots(1, 3, figsize=(11, 3.5))
for a, tau_d in zip(ax, [20, 5, 0.5]):
    m1, q, D = 3e5, 1/3, 15
    h = chirp(m1, q, D, tau_d*86400, 0.8, [antenna(0.3, 0.4, 0.5)])[0]
    f, P = spec_whitened(colored_noise()+h); a.imshow(to_image(f, P), aspect="auto", cmap="gray_r"); a.set_title(f"{tau_d} d before merger")
plt.tight_layout(); plt.show()

## 4. Generate data (takes a few minutes; lower N_TRAIN for a quick test)

In [ ]:
print("train"); Xtr, ytr, mtr = make_dataset(N_TRAIN)
print("test");  Xte, yte, mte = make_dataset(N_TEST)
print(Xtr.shape, "positives:", (ytr>0).mean())

## 5. The CNN (paper Fig. 6)
2x Conv(32) -> MaxPool -> 2x Conv(64) -> MaxPool -> 2x Conv(64) -> MaxPool -> Flatten -> Dense 64, 64, 32 -> Dense 1 (linear).

In [ ]:
def build_cnn(shape):
    i = layers.Input(shape)
    x = i
    for filters in (32, 64, 64):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Flatten()(x)
    for u in (64, 64, 32):
        x = layers.Dense(u, activation="relu")(x)
    o = layers.Dense(1)(x)
    return models.Model(i, o)

model = build_cnn(Xtr.shape[1:])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
model.summary()
hist = model.fit(Xtr, ytr, validation_split=0.1, epochs=20, batch_size=64,
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val"); plt.legend(); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.show()

## 6. Evaluation: detection rate vs. time to merger

In [ ]:
pred = model.predict(Xte, batch_size=128).ravel()
is_sig = yte > 0
print(f"False-alarm rate (noise only): {(pred[~is_sig] > 0).mean()*100:.1f}%")
print(f"Overall detection rate:        {(pred[is_sig] > 0).mean()*100:.1f}%")

tau_d = np.array([m["tau"] for m in mte])[is_sig]/86400
snr   = np.array([m["snr"] for m in mte])[is_sig]
det   = pred[is_sig] > 0
edges = np.array([0.2, 0.5, 1, 2, 5, 10, 20, 30])
centers, rate = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    s = (tau_d >= lo) & (tau_d < hi)
    if s.sum() > 5: centers.append(np.sqrt(lo*hi)); rate.append(det[s].mean()*100)
plt.semilogx(centers, rate, "o-"); plt.gca().invert_xaxis()
plt.xlabel("time to merger at window end [days]"); plt.ylabel("correct detections [%]"); plt.grid(alpha=.3); plt.show()

## 7. LISA Spritz
I could not verify the exact Spritz file layout, so this cell takes plain NumPy arrays. Get the data from the LISA Data Challenge site (account required), convert to arrays, and set the variables below.
- If you only have X, Y, Z: `A=(Z-X)/sqrt(2)`, `E=(X-2*Y+Z)/sqrt(6)`.
- Resample to 5 s if needed (the paper notes Spritz cadence is 5 s). Remove glitches/gaps as the paper did.
- The real data is in different units than my simulated noise, so the whitening reference PSD is estimated from a **signal-free stretch** of the data itself.

In [ ]:
# ---- fill these in ----
SPRITZ = None   # e.g. {"A": np.load("spritz_A.npy"), "E": np.load("spritz_E.npy")}   (1-D, 5 s cadence, glitch-free)
T_MERGER = None # merger time in seconds from the start of the arrays
NOISE_ONLY_SEG = (0, 10*86400)  # sample range [start, stop) in seconds known to contain no strong signal
# -----------------------

def run_spritz(SPRITZ, T_MERGER, NOISE_ONLY_SEG, step_h=6):
    ref = {}
    a, b = [int(s*FS) for s in NOISE_ONLY_SEG]
    for c in CHANNELS:
        f, P = signal.spectrogram(SPRITZ[c][a:b], FS, window="hann", nperseg=NPERSEG, noverlap=NOVERLAP, scaling="density", detrend=False)
        ref[c] = np.median(P, axis=1)
    ends = np.arange(N, len(SPRITZ[CHANNELS[0]])+1, int(step_h*3600*FS))
    outs = []
    for e in ends:
        imgs = []
        for c in CHANNELS:
            f, Pw = spec_whitened(SPRITZ[c][e-N:e], ref_psd=ref[c]); imgs.append(to_image(f, Pw))
        outs.append(model.predict(np.stack(imgs, -1)[None], verbose=0)[0, 0])
    ttm = (T_MERGER - ends/FS)/86400
    plt.plot(ttm, outs, "o-"); plt.axhline(0, color="k", lw=.8); plt.gca().invert_xaxis()
    plt.xlabel("time to merger [days]"); plt.ylabel("CNN output (>0 = detection)"); plt.show()
    return ttm, np.array(outs)

if SPRITZ is not None:
    ttm, outs = run_spritz(SPRITZ, T_MERGER, NOISE_ONLY_SEG)
    first = ttm[np.argmax(outs > 0)] if (outs > 0).any() else None
    print("First detection at ~", first, "days before merger (paper reports ~20 d)")